In [ ]:
import pandas as pd
import os
import glob
import traceback
import matplotlib.pyplot as plt

# Ensure plots are displayed inline in the Jupyter Notebook
%matplotlib inline

# Define the directory to search for .xlsx files (current directory)
directory = os.getcwd()

# Find all .xlsx files in the directory
all_files = glob.glob(os.path.join(directory, '*.xlsx'))

# Define a list of files to exclude
excluded_files = ['QA Log.xlsx', 'Experiment Tracker.xlsx']  # Replace with actual file names to exclude

# Filter out the excluded files, hidden files, and temporary Excel files
file_names = [os.path.splitext(os.path.basename(file))[0]  # Get the base name without extension
              for file in all_files
              if (os.path.basename(file) not in excluded_files and 
                  not os.path.basename(file).startswith('~$') and  # Exclude temporary Excel files
                  not os.path.basename(file).startswith('.'))]   # Exclude hidden files

# Define your sheet_name list
sheet_names = [
    'T0-1', 'T5-6', 'T10-11', 'T15-16', 'T20-21', 'T25-26', 'T29-30', 'T30-T31', 
    'T35-36', 'T40-41', 'T45-46', 'T50-51', 'T55-56', 'T59-60', 'T60-61', 
    'T64-65', 'T65-66', 'T70-71', 'T74-75', 'T75-76', 'T80-81', 'T84-85', 'T85-86','T90-91',
    'T120-121', 'T180-181', 'T239-240'
]

# Initialize an empty list to log failed combinations
failed_combinations = []

# Configuration: Set to True to enable plotting, False to disable
enable_plotting = False

# Function to process data for a specific file_name and sheet_name
def process_data(file_name, sheet_name):
    try:

        #!/usr/bin/env python
        # coding: utf-8

        # In[1]:


        #Import Time Points
        file_name_og=file_name
        file_name=file_name_og+'.xlsx'


        # In[2]:


        import numpy as np
        import pandas as pd
        import matplotlib.pyplot as plt
        from matplotlib.pyplot import figure
        import numpy.ma as ma
        from numpy import diff
        from scipy.ndimage import interpolation
        import math
        from sklearn import linear_model
        import statsmodels.api as sm
        from scipy.optimize import curve_fit
        import os


        # In[3]:


        time_step=0.005
        pressure_index=0
        volume_index=1
        zoom_factor=20


        # In[4]:


        #Import Raw Data
        PV_Raw = pd.read_excel (file_name, sheet_name)
        PV_Raw = PV_Raw[~PV_Raw.applymap(lambda x: isinstance(x, str)).all(axis=1)]
        # PV_Raw=PV_Raw.astype(float)


        vol=PV_Raw['LVV']

        press=PV_Raw['LVP']

        vol = np.array(PV_Raw['LVV'], dtype=float)
        press = np.array(PV_Raw['LVP'], dtype=float)


        # Calculate time array
        # time_step = 1  # You need to define your desired time step
        time_array = np.linspace(0, (len(PV_Raw) - 1) * time_step, len(PV_Raw))

        # Now 'PV_Raw' contains only rows with numeric 'LVV' and 'LVP' values, and you have the 'time_array'


        # In[5]:


        press=press-min(press)
        print(press)


        # In[ ]:





        # In[ ]:





        # In[6]:


        def RawHemodynamicPlotter (x,x_name,y,y_name,color,intensity):
            fig = plt.figure()
            fig.patch.set_facecolor(color)
            fig.patch.set_alpha(intensity)
            plt.plot(x,y, color='green')
            plt.title('{} vs {}'.format (y_name,x_name))
            plt.xlabel('{}'.format(x_name))
            plt.ylabel('{}'.format(y_name))
            plt.show()


        RawHemodynamicPlotter(vol,'Volume',press,'Pressure','pink',0.2)
        RawHemodynamicPlotter(time_array,'Time',vol,'Volume','blue',.1)
        RawHemodynamicPlotter(time_array,'Time',press,'Pressure','green',.1)


        # In[7]:


        def cart2pol(x, y):
            theta = np.arctan2(y, x)
            radius = np.sqrt(x**2 + y**2)
            return(theta, radius)


        # In[8]:


        #Define the origin from the averages of all of the points
        Pressure_origin=np.mean(press)
        Volume_origin=np.mean(vol)

        #Convert from cartesian to polar coordinates
        theta_array,radius_array=cart2pol(vol-Volume_origin,press-Pressure_origin)

        #Determine where there is a pseudo-physiological PV loops start based on arbitrary x-axis relative to the 
        PV_Loop_Start=[]
        #Define a 'for' loop over the entire length of the array to determine where the "new" PV loops start
        for step in range(1, len(radius_array)):
            if theta_array[step]>=0 and theta_array[step-1]<0:
                #Just as seen in Stonko's code, we will use an arbitrary x-axis where a shift in the signs for the theta from positive to negative values
                PV_Loop_Start.append(step)


        # In[9]:


        #Isolating individual pseudo-physiological loops
        Loops=[]
        for step in range(len(PV_Loop_Start)-1):
            #Start of new loop
            a=PV_Loop_Start[step]
            #Stop of new loop
            b=PV_Loop_Start[step+1]

            dummy_loop=np.column_stack((np.array(vol[a:b]),np.array(press[a:b])))
            Loops.append(dummy_loop)


        # In[10]:


        def centroid_calculator(loop):
            return [np.mean(loop[0]), np.mean(loop[1])]


        # In[11]:


        #Define a function to mask the loops based on what quadrant is desired
        def PV_Loop_Mask(loop,mask_value,unmasked_direction):
          #make sure first column is x values and second column is y values
            x=loop[:,0]
            y=loop[:,1]
            indices=loop[:,2]
            loop_indices=[]
            loop_x=[]
            loop_y=[]

            if unmasked_direction=='+x':
                loop=ma.masked_array(x,mask=((x<mask_value)))
            elif unmasked_direction=='-x':
                loop=ma.masked_array(x,mask=((x>mask_value)))
            elif unmasked_direction=='+y':
                loop=ma.masked_array(y,mask=((y<mask_value)))
            elif unmasked_direction=='-y':
                loop=ma.masked_array(y,mask=((y>mask_value)))    

            for i in range(len(loop)):
                if loop[i]!='--':
                    loop_indices.append(indices[i])
                    loop_x.append(x[i])
                    loop_y.append(y[i])
            loop_indices=np.array(loop_indices)
            loop_x=np.array(loop_x)
            loop_y=np.array(loop_y)
            masked_loop=np.column_stack((loop_x,loop_y,loop_indices))
            return masked_loop


        # In[12]:


        #Code to obtain the derivative of each individual sub-loop to find values such as dP/dt (and dV/dt, if needed)
        def d_dt(loop):
            derivatives=[]
            x=loop[:,0]
            y=loop[:,1]
            np.seterr(invalid='ignore')
            t=np.cumsum(np.diff(x)**2+np.diff(y)**2)
            t=np.insert(t,0,0)
            t=np.transpose(t)

            dxdt=diff(x)/diff(t)
            dydt=diff(y)/diff(t)

            derivatives=[dxdt,dydt,loop[:,2]]

            return derivatives


        # In[13]:


        def check_nth_column_legality(n,loop):
            pass_fail=0
            column_search=loop[n]
            for i in range(len(column_search)):
                if column_search[i] != 'nan' or columnn_search[i] != 0:
                    pass_fail=1
                    break
            return pass_fail


        # In[14]:


        def zoom_and_isolate(masked_loop,original_loop,zoom_factor,loop_location):
            global centroid  
            i=0
            if loop_location=='upper left':
                x_direction='-x'
                y_direction='+y'
            elif loop_location=='lower right':
                x_direction='+x'
                y_direction='-y'

            while len(masked_loop)>len(dummy_loop)/zoom_factor:
                dummy_masked_loop=[masked_loop[:,0],masked_loop[:,1]]
                centroid=centroid_calculator(dummy_masked_loop)
                dummy_masked_loop=masked_loop
                dummy_masked_loop=PV_Loop_Mask(dummy_masked_loop,centroid[0],x_direction)
                dummy_masked_loop=PV_Loop_Mask(dummy_masked_loop,centroid[1],y_direction)
                if len(dummy_masked_loop)>1:
                    size_prev=len(masked_loop)
                    size_new=len(dummy_masked_loop)
                    if size_prev==size_new:
                        break
                    else:
                        masked_loop=dummy_masked_loop
                        i=i+1
                else:
                    break
            return centroid, masked_loop


        # In[15]:


        #Emax code in case dP/dt does not work
        def corner_hunter(loop,centroid):
            dummy_loop=[loop[:,0],loop[:,1]]
            distances=[]
            counter=[]
            for i in range(len(loop)):
                x_dist=(loop[i,0]-centroid[0])**2
                y_dist=(loop[i,1]-centroid[1])**2
                dummy_distance=(x_dist+y_dist)**.5
                distances.append(dummy_distance)
                counter.append(loop[i,2])
            distances=[distances, counter]
            return distances


        # In[16]:


        EndSystoleData=[]
        EndDiastoleData=[]
        EndSystole_ddt=[]
        EndDiastole_ddt=[]


        for step in range(len(PV_Loop_Start)-1):
            # Create the dummy loop
            x=np.array(Loops[step][:,0])
            y=np.array(Loops[step][:,1])
            dummy_loop=[x,y]
            plt.plot(x,y,'cyan')
            #Get the centroid
            centroid=centroid_calculator(dummy_loop)
            plt.plot(centroid[0],centroid[1], color='purple',marker='*')


            #Create an array with all of the individual loop element values *relative to each individual loop*
            loop_element=[]
            for i in range(len(x)):
                loop_element.append(i)

            # Recreate the dummy loop to include the loop element
            dummy_loop=np.column_stack((x,y,loop_element))


            #Define the upper loop
            upper_loop=PV_Loop_Mask(dummy_loop,centroid[1],'+y')
            #Then isolate the upper left portion of the loop
            upper_left_loop=PV_Loop_Mask(upper_loop,centroid[0],'-x')

            #Define the lower loop
            lower_loop=PV_Loop_Mask(dummy_loop,centroid[1],'-y')
            #Then isolate the lower right portion of the loop
            lower_right_loop=PV_Loop_Mask(lower_loop,centroid[0],'+x')

            #Plot unprocessed upper left and lower right loops
            plt.plot(upper_left_loop[:,0],upper_left_loop[:,1],color='red')
            plt.plot(lower_right_loop[:,0],lower_right_loop[:,1],color='red')


            #Begin Zooming 
            lower_right_centroid, lower_right_masked_loop=zoom_and_isolate(lower_right_loop,dummy_loop,zoom_factor,'lower right')
            plt.plot(lower_right_centroid[0], lower_right_centroid[1], linestyle=':', marker='x', color='green')
            plt.plot(lower_right_masked_loop[:,0], lower_right_masked_loop[:,1], linestyle=':', marker='.', color='blue')    

            upper_left_centroid, upper_left_masked_loop=zoom_and_isolate(upper_left_loop,dummy_loop,zoom_factor, 'upper left')
            plt.plot(upper_left_centroid[0], upper_left_centroid[1], linestyle=':', marker='x', color='green')
            plt.plot(upper_left_masked_loop[:,0], upper_left_masked_loop[:,1], linestyle=':', marker='.', color='blue')


            #Understand upper left
            print('ESP Hunt')
            upper_left_derivatives=d_dt(upper_left_loop)
            upper_left_pass_fail=check_nth_column_legality(1,upper_left_derivatives)

            if upper_left_pass_fail==1:
                print('dp/dt')
                derivatives=d_dt(upper_left_loop)
                index=np.argmin(abs(derivatives[1]))
                ESddt=[derivatives[0][index],derivatives[1][index],derivatives[1][index]]
                EndSystole_ddt.append(ESddt)
                index=derivatives[2][index]
                index=int(index)
                hold=dummy_loop[index,:] 
                hold=hold.tolist()
                EndSystoleData.append(hold) 
            else:
                print('Emax')
                top_right_distances=corner_hunter(upper_left_loop,upper_left_centroid)
                max_dist=np.max(top_right_distances[0])
                point_max_dist=np.argmax(top_right_distances[0])
                index=int(top_right_distances[1][point_max_dist])
                hold=dummy_loop[index,:]
                ESddt=[0,0,hold]
                EndSystole_ddt.append(ESddt)
                hold=dummy_loop[int(hold[2]),:] 
                hold=hold.tolist()
                EndSystoleData.append(hold) 
            plt.plot(hold[0], hold[1], linestyle=':', marker='*', color='aqua')


            #Understand lower right
            print('EDP Hunt')
            lower_right_derivatives=d_dt(lower_right_loop)
            lower_right_pass_fail=check_nth_column_legality(1,lower_right_derivatives)
            if lower_right_pass_fail==1:
                print('dP/dt')
                derivatives=d_dt(lower_right_loop)
                index=np.argmax((derivatives[1]))
                EDddt=[derivatives[0][index],derivatives[1][index],derivatives[1][index]]
                EndDiastole_ddt.append(ESddt)
                index=derivatives[2][index]
                index=int(index)
                hold=dummy_loop[index,:] 
                hold=hold.tolist()
                EndDiastoleData.append(hold) 
            else:
                print('Emax-equivalent')
                bottom_right_distances=corner_hunter(lower_right_loop,centroid)
                if bool(bottom_right_distances[0]):
                    max_dist=np.max(bottom_right_distances[0])
                    point_max_dist=np.argmax(bottom_right_distances[0])
                    index=int(bottom_right_distances[1][point_max_dist])
                else:
                    print('Garbage')
                    total_distances=corner_hunter(dummy_loop,centroid)
                    max_dist=np.max(total_distances[0])
                    point_max_dist=np.argmax(total_distances[0])
                    index=int(total_distances[1][point_max_dist])  
                hold=dummy_loop[index,:]
                #Save this data to the End Diatole_ddt array
                EDddt=[0,0,hold]
                EndDiastole_ddt.append(ESddt)
                hold=dummy_loop[int(hold[2]),:] 
                hold=hold.tolist()
                EndDiastoleData.append(hold) 
            plt.plot(hold[0], hold[1], linestyle=':', marker='*', color='aqua')


            plt.show()


        # ### Process End Systolic and End Diastolic data to get 'real' PV Loops

        # In[17]:


        removed_list=[]
        def hunt_bad_loops(Physiological_Loops,Cycle_Start,EndSystoleData,EndDiastoleData,EndSystole_ddt,EndDiastole_ddt,Loop_Size,correction_factor, removed_list):
            #correction_factor=1

            bad_loop_hunter = input('Are there bad PV loops? Type "Yes" or "No." ')

            while bad_loop_hunter!='No' or bad_loop_hunter!='NO' or bad_loop_hunter!='no' or bad_loop_hunter!='nO':
                bad_loop_hunter = input('Enter their corresponding loop number(s). Type "No" if there are none.  ')
                if bad_loop_hunter=='No' or bad_loop_hunter=='NO' or bad_loop_hunter=='no' or bad_loop_hunter=='nO':
                    break
                removed_list.append(int(bad_loop_hunter))
                bad_loop_hunter=int(bad_loop_hunter)-correction_factor
                Cycle_Start.pop(bad_loop_hunter)
                Physiological_Loops.pop(bad_loop_hunter)
                Loop_Size.pop(bad_loop_hunter)
                EndSystoleData.pop(bad_loop_hunter)
                EndDiastoleData.pop(bad_loop_hunter)
                EndSystole_ddt.pop(bad_loop_hunter)
                EndDiastole_ddt.pop(bad_loop_hunter)
                correction_factor=correction_factor+1
                print('This number has been removed')
                #bad_loop_hunter = input('Enter their corresponding loop number(s). Type "No" if there are none remaining.')

            print('All bad loops have been removed.')
        #     for step in range(len(Physiological_Loops)):
        #         fig = plt.figure()
        #         fig.patch.set_facecolor('green')
        #         fig.patch.set_alpha(0.1)
        #         ax = fig.add_subplot(111)
        #         # Isolate an individual loop
        #         individual_loop=Physiological_Loops[step]
        #         # Convert the polar coordinates into x (volume) and y (pressure)
        #         dummy_centroid=centroid_calculator(Physiological_Loops)
        #         #plt.plot(Physiological_Loops[step][0], Physiological_Loops[step][1], linestyle=':', color='green')
        #         plt.plot(Physiological_Loops[step][0], Physiological_Loops[step][1], color='green')
        #         plt.xlabel('Volume')
        #         plt.ylabel('Pressure')
        #         plt.title('Physiological PV Loop {}'.format(step+1))
        #         plt.plot(dummy_centroid[0],dummy_centroid[1],color='red')
        #         plt.plot(EndSystoleData[step][0], EndSystoleData[step][1], linestyle=':', marker='*', color='red')
        #         plt.plot(EndDiastoleData[step][0], EndDiastoleData[step][1], linestyle=':', marker='*', color='red')

        #         plt.show()
            return Physiological_Loops,Cycle_Start,Loop_Size,EndSystoleData,EndDiastoleData,EndSystole_ddt,EndDiastole_ddt, removed_list


        # In[18]:


        PV_Loop_Start.pop(0)
        for i in range(len(EndSystoleData)-1):
            EndSystoleData[i+1][2]=EndSystoleData[i+1][2]+(PV_Loop_Start[i])
            EndDiastoleData[i+1][2]=EndDiastoleData[i+1][2]+(PV_Loop_Start[i])


        #If things get fked try commenting this out
        EndSystoleData.pop(0)
        EndDiastoleData.pop(0)
        EndSystole_ddt.pop(0)
        EndDiastole_ddt.pop(0)
        #Until here

        Cycle_Start=[]
        Physiological_Loops=[]
        Loop_Size=[]
        dummy_HR=[]
        for step in range(len(EndSystoleData)-1):
        #for step in range(len(EndSystoleData)):

        #The first loop is often never complete,so skipping it
            if step==len(EndSystoleData)-1:

                continue
            #Start of new loop
            a=int(EndSystoleData[step][2])
            a=a-1 

            #Stop of new loop
            b=int(EndSystoleData[step+1][2])
            b=b+1

            #Save the start times
            Cycle_Start.append(EndSystoleData[step][2]*time_step)

            #Obtain the Volume and Pressure Data
            volume=vol[a:b]
            volume=np.append(volume,vol[a+1])
            pressure=press[a:b]
            pressure=np.append(pressure,press[a+1])


            Loop_Size.append(len(volume))

            fig = plt.figure()
            fig.patch.set_facecolor('yellow')
            fig.patch.set_alpha(0.1)
            ax = fig.add_subplot(111)
            #Plot it
            plt.plot(volume, pressure,marker= '*')

            #Create a dummy loop to add to the list of the Physiological PV loops
            dummy_loop=[volume,pressure]
            Physiological_Loops.append(dummy_loop)

            #For Visualization purposes, show the centroid
            centroid=centroid_calculator(dummy_loop)
            plt.plot(centroid[0], centroid[1], linestyle=':', marker='*', color='red')
            plt.title('Candidate Physiological PV Loop {}'.format(step))

            plt.plot(EndSystoleData[step][0], EndSystoleData[step][1], linestyle=':', marker='*', color='red')
            plt.plot(EndDiastoleData[step][0], EndDiastoleData[step][1], linestyle=':', marker='*', color='red')

            #Show all plots
            plt.show()


            #User selection guide
            print('P-V Number of points: {}'.format(b-a+1))
            print('The Heart Rate is: {} BPM'.format(60/((b-a+1)*time_step)))
            dummy_HR.append(60/((b-a+1)*time_step))
            print('The ESV is {} and the ESP is {} at point {}.'.format(EndSystoleData[step][0], EndSystoleData[step][1], EndSystoleData[step][2]))
            if EndSystoleData[step][0]<0:
                print('Negative systolic volume detected')

            if EndSystoleData[step][1]<0:
                print('Negative systolic pressure detected')

            print('The EDV is {} and the EDP is {} at point {}.'.format(EndDiastoleData[step][0], EndDiastoleData[step][1],EndDiastoleData[step][2]))
            if EndDiastoleData[step-1][0]<0:
                print('Negative diastolic volume detected')

            if EndDiastoleData[step-1][1]<0:
                print('Negative diastolic pressure detected')


        # In[19]:


        # # if sheet_name=='T0.1' or sheet_name=='T30.1' or sheet_name=='T74.1':
        # Physiological_Loops,Cycle_Start,Loop_Size,EndSystoleData,EndDiastoleData,EndSystole_ddt,EndDiastole_ddt,removed_list=hunt_bad_loops(Physiological_Loops,Cycle_Start,EndSystoleData,EndDiastoleData,EndSystole_ddt,EndDiastole_ddt,Loop_Size,0, removed_list)


        # In[20]:


        def Calculator(time_step,Loop,Cycle_Start,Loop_Size,EndSystoleData,EndDiastoleData): #Calculate the various physiological parameters
            SW=PolyArea(Loop[0],Loop[1]) #Obtain the stroke work. This is accomplished by taking the cartesian volume of the isolated P-V loop
            SV=EndDiastoleData[0]-EndSystoleData[0] #Obtain the stroke volume, which is the difference between the end-diastolic volume and end-systolic volume.
            EJF=SV/int(EndDiastoleData[0])*100 #Obtain the ejection fraction, which is the ratio of stroke volume to EDV
            HR=60/(len(Loop[0])*time_step) #Obtain the instantaneous hear right, which is 60/(number of points in a loop* time step)
            CO=SV*HR #Obtain the cardiac output, the product of stroke volume and heart rate
            Ea=EndSystoleData[1]/SV #Obtain the arterial elastance, which is the ratio of EDP to stroke volume
            #Obtain the minimum and maximum pressures and volumes (likely not used)
            Pmax=np.max(Loop[1])
            Pmin=np.min(Loop[1])
            Vmax=np.max(Loop[0])
            Vmin=np.min(Loop[0])
            return SW,SV,EJF,HR,CO,Ea, Pmax, Pmin, Vmax, Vmin

        #Define Shoelace Formula
        def PolyArea(x_array,y_array):
            return 0.5*np.abs(np.dot(x_array,np.roll(y_array,1))-np.dot(y_array,np.roll(x_array,1)))


        # In[21]:


        # Mine the End Systolic and Diastolic Data

        #Remove the first and last points

        #The first is ignored because it is usually incomplete
        #EndSystoleData.pop(0)
        #EndDiastoleData.pop(0)

        #The last is removed because it is an end point, and usually followed by an also incomplete loop
        EndSystoleData.pop(-1)
        EndDiastoleData.pop(-1)

        ESV_total=[]
        ESP_total=[]
        EDV_total=[]
        EDP_total=[]
        dPmax=[]
        dPmin=[]



        for step in range(len(EndSystoleData)):

            ESV_total.append(EndSystoleData[step][0])
            ESP_total.append(EndSystoleData[step][1])

            EDV_total.append(EndDiastoleData[step][0])
            EDP_total.append(EndDiastoleData[step][1])

            dPmax.append(EndSystole_ddt[step][1])
            dPmin.append(EndDiastole_ddt[step][1])


        # In[22]:


        #Heart Rate pre-Calculation
        #Number of loops=number of heart beats
        number_of_loops=len(EndSystoleData)
        #Total time measured (in minutes)
        total_time=time_array[-1]/60


        # In[23]:


        #Seperate Loops into Individual Lists Elements

        StrokeWork=[]
        StrokeVolume=[]
        EjectionFraction=[]
        Heart_Rate=[]
        CardiacOutput=[]
        Elastance=[]
        MaxPressure=[]
        MinPressure=[]
        MaxVolume=[]
        MinVolume=[]


        for step in range(len(Physiological_Loops)):
            SW,SV,EJF,HR,CO,Ea, Pmax, Pmin, Vmax, Vmin=Calculator(time_step,Physiological_Loops[step],Cycle_Start,Loop_Size[step],EndSystoleData[step],EndDiastoleData[step])
            StrokeWork.append(SW)
            StrokeVolume.append(SV)
            EjectionFraction.append(EJF)
            Heart_Rate.append(HR)
            CardiacOutput.append(CO)
            Elastance.append(Ea)
            MaxPressure.append(Pmax)
            MinPressure.append(Pmin)
            MaxVolume.append(Vmax)
            MinVolume.append(Vmin)


        # In[24]:


        def HistogramGenerator(data,name):

            fig = plt.figure()
            fig.patch.set_facecolor('pink')
            fig.patch.set_alpha(0.3)
            ax = fig.add_subplot(111)

            n, bins, patches = plt.hist(x=data, bins='auto',alpha=0.6, rwidth=0.85)
            plt.grid(axis='y', alpha=0.65)
            plt.xlabel('{}'.format(name), color='red')
            plt.ylabel('Frequency', color='blue')
            plt.title('{} Frequency'.format(name), color='green')
            print('Mean {}:'.format(name))
            print(np.mean(data))

        def LinePlotter(data,name):

            fig = plt.figure()
            fig.patch.set_facecolor('purple')
            fig.patch.set_alpha(0.15)
            ax = fig.add_subplot(111)
            plt.grid(axis='y', alpha=0.65)
            plt.xlabel('Loop Number')
            plt.ylabel('{} Values'.format(name))
            #plt.xticks(np.arange(min(x), max(x)+1, 1.0))
            plt.title('{} Values as a function of PV Loop'.format(name), color='green')
            plt.plot(data,linestyle=':', marker='*', color='blue')
            plt.show()  

        def LinePlotter2(x_axis,data,name):

            fig = plt.figure()
            fig.patch.set_facecolor('green')
            fig.patch.set_alpha(0.2)
            ax = fig.add_subplot(111)
            plt.grid(axis='y', alpha=0.65)
            plt.xlabel('Time Point (seconds)')
            plt.ylabel('{} Values'.format(name))
            plt.title('{} Values as a function of Time (in seconds)'.format(name), color='green')
            plt.plot(x_axis,data,linestyle=':', marker='*', color='green')
            plt.show()
            print('This data spans over {} seconds'.format(x_axis[-1]))


        # In[25]:


        HistogramGenerator(StrokeWork,'Stroke Work (mL*mmHg)')
        LinePlotter(StrokeWork,'Stroke Work  (mL*mmHg)')
        LinePlotter2(Cycle_Start,StrokeWork,'Stroke Work  (mL*mmHg)')

        HistogramGenerator(StrokeVolume,'Stroke Volume (mL)')
        LinePlotter(StrokeVolume,'Stroke Volume (mL)')
        LinePlotter2(Cycle_Start,StrokeVolume,'Stroke Volume (mL)')

        HistogramGenerator(Elastance,'Elastance')
        LinePlotter(Elastance,'Elastance')
        LinePlotter2(Cycle_Start,Elastance,'Elastance')

        HistogramGenerator(CardiacOutput,'Cardiac Output')
        LinePlotter(CardiacOutput,'Cardiac Output')
        LinePlotter2(Cycle_Start,CardiacOutput,'Cardiac Output')

        HistogramGenerator(EjectionFraction,'Ejection Fraction')
        LinePlotter(EjectionFraction,'Ejection Fraction')
        LinePlotter2(Cycle_Start,EjectionFraction,'Ejection Fraction')

        HistogramGenerator(Heart_Rate,'Heart Rates')
        LinePlotter(Heart_Rate,'Heart Rates')
        LinePlotter2(Cycle_Start,Heart_Rate,'Heart Rates')

        HistogramGenerator(MaxPressure,'Max Pressure (mmHg)')
        LinePlotter(MaxPressure,'Max Pressure (mmHg)')
        LinePlotter2(Cycle_Start,MaxPressure,'Max Pressure (mmHg)')

        HistogramGenerator(MinPressure,'Min Pressure (mmHg)')
        LinePlotter(MinPressure,'Min Pressure (mmHg)')
        LinePlotter2(Cycle_Start,MinPressure,'Min Pressure (mmHg)')

        HistogramGenerator(MaxVolume,'Max Volume (mL)')
        LinePlotter(MaxVolume,'Max Volume (mL)')
        LinePlotter2(Cycle_Start,MaxVolume,'Max Volume (mL)')

        HistogramGenerator(MinVolume,'Min Volume (mL)')
        LinePlotter(MinVolume,'Min Volume (mL)')
        LinePlotter2(Cycle_Start,MinVolume,'Min Volume (mL)')

        HistogramGenerator(ESV_total,'ESV (mL)')
        LinePlotter(ESV_total,'ESV (mL)')
        LinePlotter2(Cycle_Start,ESV_total,'ESV (mL)')

        HistogramGenerator(ESP_total,'ESP (mmHg)')
        LinePlotter(ESP_total,'ESP (mmHg)')
        LinePlotter2(Cycle_Start,ESP_total,'ESP (mmHg)')

        HistogramGenerator(EDV_total,'EDV (mL)')
        LinePlotter(EDV_total,'EDV (mL)')
        LinePlotter2(Cycle_Start,EDV_total,'EDV (mL)')

        HistogramGenerator(EDP_total,'EDP (mmHg)')
        LinePlotter(EDP_total,'EDP (mmHg)')
        LinePlotter2(Cycle_Start,EDP_total,'EDP (mmHg)')


        # In[26]:


        print('Raw Mean Data Information')

        rawmean_SW=np.mean(StrokeWork)
        print('The Raw Mean of Stroke Work is {}'.format(rawmean_SW))

        rawmean_SV=np.mean(StrokeVolume)
        print('The Raw Mean of the Stroke Volume is {}'.format(rawmean_SV))

        rawmean_EJF=np.mean(EjectionFraction)
        print('The Raw Mean of the Ejection Fraction is {}'.format(rawmean_EJF))

        rawmean_HR=np.mean(Heart_Rate)
        print('The Raw Mean of the Heart Rate is {}'.format(rawmean_HR))

        rawmean_CO=np.mean(CardiacOutput)
        print('The Raw Mean of the Cardiac Output is {}'.format(rawmean_CO))

        rawmean_Elastance=np.mean(Elastance)
        print('The Raw Mean of the Elastance is {}'.format(rawmean_Elastance))

        rawmean_MaxPressure=np.mean(list(MaxPressure))
        print('The Raw Mean of the Max Pressure is {}'.format(rawmean_MaxPressure))

        rawmean_MinPressure=np.mean(list(MinPressure))
        print('The Raw Mean of the Min Pressure is {}'.format(rawmean_MinPressure))

        rawmean_MaxVolume=np.mean(list(MaxVolume))
        print('The Raw Mean of the Max Volume is {}'.format(rawmean_MaxVolume))

        rawmean_MinVolume=np.mean(list(MinVolume))
        print('The Raw Mean of the Min Volume is {}'.format(rawmean_MinVolume))

        rawmean_ESV=np.mean(list(ESV_total))
        print('The Raw Mean of the ESV is {}'.format(rawmean_ESV))

        rawmean_ESP=np.mean(list(ESP_total))
        print('The Raw Mean of the ESP is {}'.format(rawmean_ESP))

        rawmean_EDV=np.mean(list(EDV_total))
        print('The Raw Mean of the EDV is {}'.format(rawmean_EDV))

        rawmean_EDP=np.mean(list(EDP_total))
        print('The Raw Mean of the EDP is {}'.format(rawmean_EDP))


        # In[27]:


        def ResizeArray (np_array, desired_size):
            z = desired_size / len(np_array)
            reshaped_np_array = interpolation.zoom(np_array,z)
            return reshaped_np_array


        # In[28]:


        #Code to obtain the ensemble average
        avg_loop_size=np.mean(Loop_Size)
        avg_loop_size=np.rint(avg_loop_size)

        max_vol=[]
        min_vol=[]
        max_press=[]
        min_press=[]

        ensemble_volume=np.zeros(shape=(int(avg_loop_size)-1,len(Physiological_Loops)))
        ensemble_pressure=np.zeros(shape=(int(avg_loop_size)-1,len(Physiological_Loops)))



        for step in range(len(Physiological_Loops)):
            x=np.array(Physiological_Loops[step])
            dummy_volume=ResizeArray (x[0], avg_loop_size-1)
            ensemble_volume[:,step]=dummy_volume
            dummy_pressure=ResizeArray (x[1], avg_loop_size-1)
            ensemble_pressure[:,step]=dummy_pressure



        for step in range(len(ensemble_volume)):
            max_vol.append(np.max(ensemble_volume[step]))
            min_vol.append(np.min(ensemble_volume[step]))
            max_press.append(np.max(ensemble_pressure[step]))
            min_press.append(np.min(ensemble_pressure[step]))


        #Determine the ensemble average from all of this data

        ensemble_volume=np.mean(ensemble_volume, axis=1)
        ensemble_pressure=np.mean(ensemble_pressure, axis=1)

        ensemble_volume = ensemble_volume[:-1]
        ensemble_pressure = ensemble_pressure[:-1]


        #Close the loop
        ensemble_volume=np.append(ensemble_volume,ensemble_volume[0])
        ensemble_pressure=np.append(ensemble_pressure,ensemble_pressure[0])




        #Resize
        ensemble_volume=ResizeArray (ensemble_volume, avg_loop_size)
        ensemble_pressure=ResizeArray (ensemble_pressure, avg_loop_size)

        for i in range(len(ensemble_volume)):
            if ensemble_volume[i]==0 and ensemble_pressure[i]==0:
                ensemble_volume=np.delete(ensemble_volume,i)
                ensemble_pressure=np.delete(ensemble_pressure,i)


        ensemble_loop=[ensemble_volume,ensemble_pressure]

        for i in range(len(ensemble_loop)):
            if ensemble_loop[i][0]==0 and ensemble_loop[i][1]==0:
                ensemble_loop.pop(i)

        ESPV_ensemble=[]
        EDPV_ensemble=[]


        # In[29]:


        fig = plt.figure()
        fig.patch.set_facecolor('blue')
        fig.patch.set_alpha(0.1)
        ax = fig.add_subplot(111)


        ESPV_ensemble=[]
        EDPV_ensemble=[]

        zoom_factor=8
        #Create an array with all of the individual loop element values *relative to each individual loop*

        centroid=centroid_calculator(ensemble_loop)

        loop_element=[]
        for i in range(len(ensemble_volume)):
            loop_element.append(i)
        # Recreate the dummy loop to include the loop element
        dummy_loop=np.column_stack((ensemble_loop[0],ensemble_loop[1],loop_element))




        # Create the dummy loop
        x=np.array(ensemble_loop[0])
        y=np.array(ensemble_loop[1])
        plt.plot(x,y,color='green')
        dummy_loop=[x,y]

        #Get the centroid
        centroid=centroid_calculator(dummy_loop)
        plt.plot(centroid[0],centroid[1], color='purple',marker='*')
        #Create an array with all of the individual loop element values *relative to each individual loop*
        loop_element=[]
        for i in range(len(x)):
            loop_element.append(i)

        # Recreate the dummy loop to include the loop element
        dummy_loop=np.column_stack((x,y,loop_element))


        #Define the upper loop
        upper_loop=PV_Loop_Mask(dummy_loop,centroid[1],'+y')
        #Then isolate the upper left portion of the loop
        upper_left_loop=PV_Loop_Mask(upper_loop,centroid[0],'-x')

        #Define the lower loop
        lower_loop=PV_Loop_Mask(dummy_loop,centroid[1],'-y')
        #Then isolate the lower right portion of the loop
        lower_right_loop=PV_Loop_Mask(lower_loop,centroid[0],'+x')

        #Plot unprocessed upper left and lower right loops
        plt.plot(upper_left_loop[:,0],upper_left_loop[:,1],color='red')
        plt.plot(lower_right_loop[:,0],lower_right_loop[:,1],color='red')

        #To see if the dP/dt algorithm will work by checking the second (n=1) column
        upper_left_derivatives=d_dt(upper_left_loop)
        upper_left_pass_fail=check_nth_column_legality(1,upper_left_derivatives)

        lower_right_derivatives=d_dt(lower_right_loop)
        lower_right_pass_fail=check_nth_column_legality(1,lower_right_derivatives)

        #Begin Zooming
        zoom_factor=len(dummy_loop)/zoom_factor

        ###############################################################################################    
        #Zoom the upper left side
        i=0

        while len(upper_left_loop)>len(dummy_loop)/zoom_factor:
            dummy_upper_left_loop=[upper_left_loop[:,0],upper_left_loop[:,1]]
            centroid2=centroid_calculator(dummy_upper_left_loop)
            dummy_upper_left_loop=upper_left_loop
            dummy_upper_left_loop=PV_Loop_Mask(dummy_upper_left_loop,centroid2[0],'-x')
            dummy_upper_left_loop=PV_Loop_Mask(dummy_upper_left_loop,centroid2[1],'+y')
            if len(dummy_upper_left_loop)>1:
                size_prev=len(upper_left_loop)
                size_new=len(dummy_upper_left_loop)
                if size_prev==size_new:
                    break
                else:
                    upper_left_loop=dummy_upper_left_loop
                    upper_left_centroid=centroid2
                    i=i+1
            else:
                break
        plt.plot(upper_left_centroid[0], upper_left_centroid[1], linestyle=':', marker='x', color='aqua')
        print('The upper left loop was zoomed {} times'.format(i)) 
        plt.plot(upper_left_loop[:,0], upper_left_loop[:,1], linestyle=':', marker='.', color='blue') 

        if upper_left_pass_fail==1:
            print('Angel')
            derivatives=d_dt(upper_left_loop)

            #Note- making an assumption it is abs value
            #index=np.argmax(abs(derivatives[1]))
            index=np.argmin((derivatives[1]))
            #Save this data to the End Systole_ddt array
            ESddt=[derivatives[0][index],derivatives[1][index],derivatives[1][index]]
            #EndSystole_ddt.append(ESddt)


            index=derivatives[2][index]
            index=int(index)

            hold=dummy_loop[index,:] 

            hold=hold.tolist()
            ESPV_ensemble.append(hold) 
        else:
            print('Devil')
            top_right_distances=corner_hunter(upper_left_loop,upper_left_centroid)
            max_dist=np.max(top_right_distances[0])
            point_max_dist=np.argmax(top_right_distances[0])
            index=int(top_right_distances[1][point_max_dist])

            hold=dummy_loop[index,:]
            #Save this data to the End Systole_ddt array
            ESddt=[0,0,hold]
            #EndSystole_ddt.append(ESddt)
            hold=dummy_loop[int(hold[2]),:] 
            hold=hold.tolist()
            ESPV_ensemble.append(hold) 
        plt.plot(hold[0], hold[1], linestyle=':', marker='*', color='aqua')

        ###############################################################################################





        ###############################################################################################    
        #Zoom the lower right side
        i=0

        while len(lower_right_loop)>len(dummy_loop)/zoom_factor:
            dummy_lower_right_loop=[lower_right_loop[:,0],lower_right_loop[:,1]]
            centroid2=centroid_calculator(dummy_lower_right_loop)

            dummy_lower_right_loop=lower_right_loop
            dummy_lower_right_loop=PV_Loop_Mask(dummy_lower_right_loop,centroid2[0],'+x')
            dummy_lower_right_loop=PV_Loop_Mask(dummy_lower_right_loop,centroid2[1],'-y')
            if len(dummy_lower_right_loop)>1:
                size_prev=len(lower_right_loop)
                size_new=len(dummy_lower_right_loop)
                if size_prev==size_new:
                    break
                else:
                    lower_right_loop=dummy_lower_right_loop
                    i=i+1
                    lower_right_centroid=centroid2

            else:    
                break
        plt.plot(lower_right_centroid[0], lower_right_centroid[1], linestyle=':', marker='x', color='aqua')
        print('The lower right loop was zoomed {} times'.format(i)) 
        plt.plot(lower_right_loop[:,0], lower_right_loop[:,1], linestyle=':', marker='.', color='blue')


        if lower_right_pass_fail==1:
            print('Angel')
            derivatives=d_dt(lower_right_loop)

            #Note- making an assumption it is abs value
            #index=np.argmax(abs(derivatives[1]))
            index=np.argmax((derivatives[1]))
            #Save this data to the End Systole_ddt array
            EDddt=[derivatives[0][index],derivatives[1][index],derivatives[1][index]]
            #EndDiastole_ddt.append(ESddt)


            index=derivatives[2][index]
            index=int(index)

            hold=dummy_loop[index,:] 
            hold=hold.tolist()
            EDPV_ensemble.append(hold) 
        else:
            print('Devil')
            bottom_right_distances=corner_hunter(lower_right_loop,centroid)
            max_dist=np.max(bottom_right_distances[0])
            point_max_dist=np.argmax(bottom_right_distances[0])
            index=int(bottom_right_distances[1][point_max_dist])
            hold=dummy_loop[index,:]
            #Save this data to the End Diatole_ddt array
            EDddt=[0,0,hold]
            #EndDiastole_ddt.append(ESddt)

            hold=dummy_loop[int(hold[2]),:] 
            hold=hold.tolist()
            EDPV_ensemble.append(hold) 
        plt.plot(hold[0], hold[1], linestyle=':', marker='*', color='aqua')

        ###############################################################################################

        plt.show()


        # In[30]:


        SW_ensemble,SV_ensemble,EJF_ensemble,HR_ensemble,CO_ensemble,Elastance_ensemble,Pmax_ensemble, Pmin_ensemble, Vmax_ensemble, Vmin_ensemble=Calculator(time_step,ensemble_loop,Cycle_Start,len(ensemble_loop),ESPV_ensemble[0],EDPV_ensemble[0])


        # In[31]:


        import os


        # In[ ]:





        # In[ ]:





        # In[32]:


        output_file_name=file_name+'_'+sheet_name

        titles=['Stroke Work', 'Stroke Volume' , 'Ejection Fraction', 'Heart Rate', 'Cardiac Output', 'Elastance', 'Max Pressure', 'Min Pressure', 'Max Volume', 'Min Volume', 'ESV', 'ESP', 'EDV', 'EDP']

        rawmean=np.array([rawmean_SW,rawmean_SV,rawmean_EJF,rawmean_HR,rawmean_CO,rawmean_Elastance,rawmean_MaxPressure,rawmean_MinPressure,rawmean_MaxVolume,rawmean_MinVolume,rawmean_ESV,rawmean_ESP,rawmean_EDV,rawmean_EDP])
        Output_rawmean= pd.DataFrame(rawmean)
        Output_rawmean=Output_rawmean.T
        Output_rawmean.columns=titles


        Ensemble_Hemodata=np.array([SW_ensemble,SV_ensemble,EJF_ensemble,HR_ensemble,CO_ensemble,Elastance_ensemble,Pmax_ensemble, Pmin_ensemble, Vmax_ensemble, Vmin_ensemble, ESPV_ensemble[0][0],ESPV_ensemble[0][1],EDPV_ensemble[0][0],EDPV_ensemble[0][1]])
        Output_Ensemble_Hemodata= pd.DataFrame(Ensemble_Hemodata)
        Output_Ensemble_Hemodata=Output_Ensemble_Hemodata.T
        Output_Ensemble_Hemodata.columns=titles
        Output_Ensemble_Hemodata= pd.DataFrame(Output_Ensemble_Hemodata)



        #pure_raw_data=np.zeros((int(len(StrokeWork)-1),int(len(titles)-1)))

        #for step in range (len(StrokeWork))
        pure_raw_data=np.vstack([StrokeWork, StrokeVolume, EjectionFraction, Heart_Rate, CardiacOutput, Elastance, MaxPressure, MinPressure, MaxVolume, MinVolume, ESV_total, ESP_total, EDV_total, EDP_total])
        #pure_raw_data=np.vstack([StrokeWork, StrokeVolume, EjectionFraction, Heart_Rate, CardiacOutput, Elastance, MaxPressure, MinPressure, MaxVolume, MinVolume])
        pure_raw_data= pd.DataFrame(pure_raw_data)
        pure_raw_data=pure_raw_data.T
        #titles2=['Stroke Work', 'Stroke Volume' , 'Ejection Fraction', 'Heart Rate', 'Cardiac Output', 'Elastance', 'Max Pressure', 'Min Pressure', 'Max Volume', 'Min Volume']
        pure_raw_data.columns=titles
        pure_raw_data= pd.DataFrame(pure_raw_data)


        Ensemble_Frame=[ensemble_pressure,ensemble_volume]
        Ensemble_Frame=np.array(Ensemble_Frame)
        Ensemble_Frame=np.transpose(Ensemble_Frame)
        Ensemble_Frame= pd.DataFrame(Ensemble_Frame, columns = ['Pressure', 'Volume'] )

        trash_loops=pd.DataFrame(np.transpose(np.array(removed_list)))

        # os.makedirs(os.path.join(file_name_og, sheet_name), exist_ok=True)


        # with pd.ExcelWriter("{}_T{}_PV_Output.xlsx".format(file_name_og,sheet_name)) as writer:
        #     Output_rawmean.to_excel(writer, sheet_name="Raw Mean Data", index=0)
        #     Output_Ensemble_Hemodata.to_excel(writer, sheet_name="Ensemble Data", index=0)
        #     pure_raw_data.to_excel(writer, sheet_name="All Raw Data", index=0)
        #     Ensemble_Frame.to_excel(writer, sheet_name="Ensemble PV Loop", index=0)
        #     trash_loops.to_excel(writer, sheet_name="Discarded Loops", index=0, header=False)


        # In[33]:


        # Define the directory based on os.makedirs
        output_directory = os.path.join(file_name_og, sheet_name)

        # Create the directory if it doesn't exist
        os.makedirs(output_directory, exist_ok=True)

        # Define the file path
        file_path = os.path.join(output_directory, "{}_T{}_PV_Output.xlsx".format(file_name_og, sheet_name))

        # Use pd.ExcelWriter with the full file path
        with pd.ExcelWriter(file_path) as writer:
            Output_rawmean.to_excel(writer, sheet_name="Raw Mean Data", index=0)
            Output_Ensemble_Hemodata.to_excel(writer, sheet_name="Ensemble Data", index=0)
            pure_raw_data.to_excel(writer, sheet_name="All Raw Data", index=0)
            Ensemble_Frame.to_excel(writer, sheet_name="Ensemble PV Loop", index=0)
            trash_loops.to_excel(writer, sheet_name="Discarded Loops", index=0, header=False)


        # In[34]:


        import os
        import matplotlib.pyplot as plt
        import pandas as pd
        import numpy as np

        # Define a function to save the graph
        def save_graph(data, name, output_directory):
            fig = plt.figure()
            fig.patch.set_facecolor('pink')
            fig.patch.set_alpha(0.3)
            ax = fig.add_subplot(111)

            n, bins, patches = plt.hist(x=data, bins='auto', alpha=0.6, rwidth=0.85)
            plt.grid(axis='y', alpha=0.65)
            plt.xlabel('{}'.format(name), color='red')
            plt.ylabel('Frequency', color='blue')
            plt.title('{} Frequency'.format(name), color='green')
            print('Mean {}:'.format(name))
            print(np.mean(data))

            # Save the graph as an image
            graph_file_path = os.path.join(output_directory, '{}_Histogram.png'.format(name))
            plt.savefig(graph_file_path, dpi=300, bbox_inches='tight')

        # Define a function to save line plots
        def save_line_plot(data, name, output_directory):
            fig = plt.figure()
            fig.patch.set_facecolor('purple')
            fig.patch.set_alpha(0.15)
            ax = fig.add_subplot(111)
            plt.grid(axis='y', alpha=0.65)
            plt.xlabel('Loop Number')
            plt.ylabel('{} Values'.format(name))
            plt.title('{} Values as a function of PV Loop'.format(name), color='green')
            plt.plot(data, linestyle=':', marker='*', color='blue')

            # Save the line plot as an image
            plot_file_path = os.path.join(output_directory, '{}_LinePlot.png'.format(name))
            plt.savefig(plot_file_path, dpi=300, bbox_inches='tight')

        # Define a function to save LinePlotter2 graphs
        def save_line_plot2(x_axis, data, name, output_directory):
            fig = plt.figure()
            fig.patch.set_facecolor('green')
            fig.patch.set_alpha(0.2)
            ax = fig.add_subplot(111)
            plt.grid(axis='y', alpha=0.65)
            plt.xlabel('Time Point (seconds)')
            plt.ylabel('{} Values'.format(name))
            plt.title('{} Values as a function of Time (in seconds)'.format(name), color='green')
            plt.plot(x_axis, data, linestyle=':', marker='*', color='green')
            print('This data spans over {} seconds'.format(x_axis[-1]))

            # Save the LinePlotter2 graph as an image
            plot2_file_path = os.path.join(output_directory, '{}_LinePlot2.png'.format(name))
            plt.savefig(plot2_file_path, dpi=300, bbox_inches='tight')

        # Create the directory based on os.makedirs
        output_directory = os.path.join(file_name_og, sheet_name)
        os.makedirs(output_directory, exist_ok=True)

        # Define a list of data and their corresponding names
        data_list = [(StrokeWork, 'StrokeWork'), (StrokeVolume, 'StrokeVolume'), (Elastance, 'Elastance'),
                     (CardiacOutput, 'CardiacOutput'), (EjectionFraction, 'EjectionFraction'),
                     (Heart_Rate, 'Heart_Rate'), (MaxPressure, 'MaxPressure'), (MinPressure, 'MinPressure'),
                     (MaxVolume, 'MaxVolume'), (MinVolume, 'MinVolume'), (ESV_total, 'ESV_total'),
                     (ESP_total, 'ESP_total'), (EDV_total, 'EDV_total'), (EDP_total, 'EDP_total')]

        for data, name in data_list:
            # Save the histograms
            save_graph(data, name, output_directory)

            # Save the line plots
            save_line_plot(data, name, output_directory)

            # Save LinePlotter2 graphs
            save_line_plot2(Cycle_Start, data, name, output_directory)

        # Define the file path for the Excel file
        excel_file_path = os.path.join(output_directory, "{}_T{}_PV_Output.xlsx".format(file_name_og, sheet_name))

        # Use pd.ExcelWriter with the full file path
        with pd.ExcelWriter(excel_file_path) as writer:
            Output_rawmean.to_excel(writer, sheet_name="Raw Mean Data", index=0)
            Output_Ensemble_Hemodata.to_excel(writer, sheet_name="Ensemble Data", index=0)
            pure_raw_data.to_excel(writer, sheet_name="All Raw Data", index=0)
            Ensemble_Frame.to_excel(writer, sheet_name="Ensemble PV Loop", index=0)
            trash_loops.to_excel(writer, sheet_name="Discarded Loops", index=0, header=False)

        # # Show the graphs (optional)
        # plt.show()


        # In[35]:


        import os
        import matplotlib.pyplot as plt

        # Define the directory based on os.makedirs
        output_directory = os.path.join(file_name_og, sheet_name)

        # Create the directory if it doesn't exist
        os.makedirs(output_directory, exist_ok=True)

        # Define the file path for the Excel file
        excel_file_path = os.path.join(output_directory, "{}_T{}_PV_Output.xlsx".format(file_name_og, sheet_name))

        # Define the file path for the graph
        graph_file_path = os.path.join(output_directory, "Ensemble Pressure_vs_Volume.png")

        # Create the graph
        fig = plt.figure()
        fig.patch.set_facecolor('blue')
        fig.patch.set_alpha(0.05)
        ax = fig.add_subplot(111)

        final_number_of_loops = len(Physiological_Loops)

        plt.plot(vol, press, color='violet')
        plt.plot(ensemble_volume, ensemble_pressure, color='green')

        plt.xlabel('Volume', color='black')
        plt.ylabel('Pressure', color='black')
        plt.title('Pressure vs Volume of All Loops ({} loops)'.format(final_number_of_loops), color='black')

        ax.legend(['Raw Data', 'Ensemble Average'])

        # Save the graph as an image
        plt.savefig(graph_file_path, dpi=300, bbox_inches='tight')

        # Save the Excel file
        with pd.ExcelWriter(excel_file_path) as writer:
            Output_rawmean.to_excel(writer, sheet_name="Raw Mean Data", index=0)
            Output_Ensemble_Hemodata.to_excel(writer, sheet_name="Ensemble Data", index=0)
            pure_raw_data.to_excel(writer, sheet_name="All Raw Data", index=0)
            Ensemble_Frame.to_excel(writer, sheet_name="Ensemble PV Loop", index=0)
            trash_loops.to_excel(writer, sheet_name="Discarded Loops", index=0, header=False)

        # Show the graph (optional)
        plt.show()


        # In[36]:


        # final_number_of_loops = len(Physiological_Loops)

        # plt.plot(vol, press, color='violet')
        # plt.plot(ensemble_volume, ensemble_pressure, color='green')

        # plt.xlabel('Volume', color='black')
        # plt.ylabel('Pressure', color='black')
        # plt.title('Pressure vs Volume of All Loops ({} loops)'.format(final_number_of_loops), color='black')

        # ax.legend(['Raw Data', 'Ensemble Average'])


        # In[37]:


        # print(Physiological_Loops)
        print(Physiological_Loops[0][0])


        # In[38]:


        i=0
        # Plot all loops on the same graph
        for loop in Physiological_Loops:
            x_values = loop[0]
            y_values = loop[1]
            i+=1
            plt.plot(x_values, y_values, color='red')

        # # Add labels and legend
        # plt.xlabel('X-axis')
        # plt.ylabel('Y-axis')
        # plt.title('Physiological Loops')
        # plt.legend()

        plt.plot(ensemble_volume, ensemble_pressure, color='green')

        plt.xlabel('Volume', color='black')
        plt.ylabel('Pressure', color='black')
        plt.title('Pressure vs Volume of All Loops ({} loops)'.format(final_number_of_loops), color='black')

        ax.legend(['Raw Data', 'Ensemble Average'])

        # Show the plot
        plt.show()



        
        
        

    except Exception as e:
        print(f"Error processing {file_name}.xlsx - {sheet_name}: {e}")
        traceback.print_exc()
        failed_combinations.append((file_name, sheet_name))

# Loop through each file_name and sheet_name and process it
for file_name in file_names:
    for sheet_name in sheet_names:
        try:
            process_data(file_name, sheet_name)
        except Exception as e:
            print(f"Failed to process {file_name}.xlsx - {sheet_name}: {e}")
            traceback.print_exc()
            failed_combinations.append((file_name, sheet_name))

# # Print the log of failed combinations
# if failed_combinations:
#     print("The following combinations failed:")
#     for combo in failed_combinations:
#         print(combo)
# else:
#     print("All combinations processed successfully!")
# Check if there are any failed combinations


In [ ]:
import datetime
if failed_combinations:
    # Get the current date and time
    now = datetime.datetime.now()
    # Format the filename with the current date and time
    filename = f"Error Report {now.strftime('%Y-%m-%d %H-%M-%S')}.txt"
    
    # Open the file and write the failed combinations
    with open(filename, 'w') as file:
        file.write("The following combinations failed:\n")
        for combo in failed_combinations:
            file.write(f"{combo}\n")
    
    print(f"Error report written to {filename}")
else:
    print("All combinations processed successfully!")